
# CardioNexus: Multimodal ECG–PPG Intelligence for Reliable Real-Time Atrial Fibrillation Screening at the Edge

## Full Google Colab implementation — low-compute edition

This notebook is designed for a **CPU-first Google Colab run**. It uses only the scientific Python stack already present in Colab:
**NumPy, pandas, SciPy, scikit-learn, matplotlib, requests and joblib**.

No TensorFlow, PyTorch, XGBoost, LightGBM, WFDB, SHAP, or other binary packages are required.  
This deliberately reduces package conflicts, setup time, GPU dependence, and Colab runtime errors.

### Dataset
**MIMIC PERform AF Dataset**
- 35 critically ill adults
- 19 AF subjects and 16 non-AF subjects
- ECG + PPG + respiration
- 20-minute recordings
- 125 Hz sampling
- CSV distribution from Zenodo

### Scientific endpoint
This notebook performs **AF vs non-AF screening**. It does **not** claim general cardiovascular-disease diagnosis.

### Advanced but low-compute learning strategy
1. ECG HRV/rhythm features
2. PPG pulse/rhythm features
3. ECG–PPG cross-signal features
4. ExtraTrees
5. Random Forest
6. Histogram Gradient Boosting
7. Validation-weighted multi-learner fusion
8. Probability calibration
9. Split-conformal prediction
10. Permutation-based feature attribution
11. Sensor-missingness and noise robustness
12. Subject-level bootstrap confidence intervals
13. McNemar significance test
14. Edge inference latency, throughput and model size

### Leakage control
All splits are **subject-wise**. Windows from one patient cannot appear in both training and test sets.


In [ ]:

#@title 1. Imports and reproducibility — no pip installation required
import os, re, json, time, math, random, zipfile, hashlib, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from scipy import signal, stats
from scipy.special import expit
from scipy.optimize import minimize_scalar

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, log_loss, brier_score_loss
)
from sklearn.inspection import permutation_importance
import joblib

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FS = 125
WINDOW_SEC = 30
STEP_SEC = 30
MAX_WINDOWS_PER_SUBJECT = 20
BOOTSTRAPS = 1000

ROOT = Path("/content/cardionexus")
DATA_DIR = ROOT / "data"
OUT = ROOT / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

print("Environment ready")
print("Sampling rate:", FS, "Hz")
print("Window:", WINDOW_SEC, "s")
print("Maximum windows per subject:", MAX_WINDOWS_PER_SUBJECT)


In [ ]:

#@title 2. Robust download of the official MIMIC PERform AF CSV archives
ZENODO_BASE = "https://zenodo.org/records/6807403/files"

FILES = {
    "AF": {
        "name": "mimic_perform_af_csv.zip",
        "md5": "df323c2be9db41589b5011ac9efb54ca",
    },
    "NON_AF": {
        "name": "mimic_perform_non_af_csv.zip",
        "md5": "84d5ad5e9fe94e83b40d6dbd36e4210e",
    }
}

session = requests.Session()
adapter = requests.adapters.HTTPAdapter(max_retries=8)
session.mount("https://", adapter)

def md5sum(path, chunk=1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def robust_download(url, dst, expected_md5=None, attempts=5):
    dst = Path(dst)
    if dst.exists() and dst.stat().st_size > 100_000:
        if expected_md5 is None or md5sum(dst) == expected_md5:
            print("Using cached:", dst.name)
            return dst

    tmp = Path(str(dst) + ".part")
    for attempt in range(1, attempts + 1):
        try:
            if tmp.exists():
                tmp.unlink()

            with session.get(url, stream=True, timeout=(20, 180)) as r:
                r.raise_for_status()
                total = int(r.headers.get("content-length", 0))
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1024*512):
                        if chunk:
                            f.write(chunk)

            if tmp.stat().st_size < 100_000:
                raise IOError("Downloaded archive is unexpectedly small")

            if total and tmp.stat().st_size != total:
                raise IOError(
                    f"Partial download: got {tmp.stat().st_size}, expected {total} bytes"
                )

            if expected_md5 and md5sum(tmp) != expected_md5:
                raise IOError("MD5 mismatch; archive is incomplete/corrupt")

            os.replace(tmp, dst)
            print("Downloaded and validated:", dst.name)
            return dst

        except Exception as e:
            print(f"Attempt {attempt}/{attempts} failed for {dst.name}: {e}")
            if tmp.exists():
                tmp.unlink()
            time.sleep(min(30, 2**attempt))

    raise RuntimeError(
        f"Could not download {dst.name} after {attempts} attempts. "
        "Rerun only this cell later; already validated archives are reused."
    )

archives = {}
for label, info in FILES.items():
    url = f"{ZENODO_BASE}/{info['name']}?download=1"
    archives[label] = robust_download(
        url, DATA_DIR / info["name"], info["md5"]
    )

print("Both official archives are ready.")


In [ ]:

#@title 3. Extract archives and discover CSV recordings
extract_roots = {}

for label, archive in archives.items():
    dst = DATA_DIR / label
    dst.mkdir(exist_ok=True)
    marker = dst / ".extracted_ok"

    if not marker.exists():
        for p in dst.iterdir():
            if p.is_dir():
                shutil.rmtree(p)
            else:
                p.unlink()

        with zipfile.ZipFile(archive, "r") as z:
            bad = z.testzip()
            if bad is not None:
                raise RuntimeError(f"Corrupt ZIP member detected: {bad}")
            z.extractall(dst)

        marker.write_text("ok")

    extract_roots[label] = dst

csv_map = {}
for label, root in extract_roots.items():
    files = sorted([p for p in root.rglob("*.csv") if p.is_file()])
    csv_map[label] = files
    print(label, "CSV files:", len(files))
    if files:
        print(" example:", files[0])

assert len(csv_map["AF"]) >= 10, "Too few AF CSV files were discovered."
assert len(csv_map["NON_AF"]) >= 10, "Too few non-AF CSV files were discovered."


In [ ]:

#@title 4. Robust ECG/PPG column audit
def norm_col(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

def pick_col(columns, kind):
    cols = list(columns)
    norm = [norm_col(c) for c in cols]

    if kind == "ECG":
        exact = ["ecg", "ii", "leadii"]
        contains = ["ecg"]
    elif kind == "PPG":
        exact = ["ppg", "pleth", "photoplethysmogram"]
        contains = ["ppg", "pleth"]
    else:
        return None

    for target in exact:
        if target in norm:
            return cols[norm.index(target)]

    for i, n in enumerate(norm):
        if any(k in n for k in contains):
            return cols[i]

    return None

audit_rows = []
usable = []

for label, paths in csv_map.items():
    for p in paths:
        try:
            head = pd.read_csv(p, nrows=10)
            ecg_col = pick_col(head.columns, "ECG")
            ppg_col = pick_col(head.columns, "PPG")

            ok = ecg_col is not None and ppg_col is not None
            audit_rows.append({
                "class": label,
                "file": str(p),
                "subject": p.stem,
                "columns": " | ".join(map(str, head.columns)),
                "ecg_column": ecg_col if ecg_col else "",
                "ppg_column": ppg_col if ppg_col else "",
                "status": "OK" if ok else "MISSING_ECG_OR_PPG"
            })

            if ok:
                usable.append({
                    "label": 1 if label == "AF" else 0,
                    "class": label,
                    "path": p,
                    "subject": p.stem,
                    "ecg_col": ecg_col,
                    "ppg_col": ppg_col
                })

        except Exception as e:
            audit_rows.append({
                "class": label,
                "file": str(p),
                "subject": p.stem,
                "status": f"READ_ERROR: {e}"
            })

audit_df = pd.DataFrame(audit_rows)
display(audit_df[["class","subject","ecg_column","ppg_column","status"]])
audit_df.to_csv(OUT / "dataset_audit.csv", index=False)

usable_df = pd.DataFrame(usable)
display(usable_df.groupby("class")["subject"].nunique())

assert usable_df["subject"].nunique() >= 25, (
    "Fewer than 25 usable ECG+PPG subjects were detected. "
    "Inspect dataset_audit.csv before continuing."
)


In [ ]:

#@title 5. Signal preprocessing and low-cost physiological feature functions
def finite_interp(x):
    x = np.asarray(x, dtype=float)
    x[~np.isfinite(x)] = np.nan
    return pd.Series(x).interpolate(limit_direction="both").bfill().ffill().to_numpy(dtype=float)

def bandpass(x, low, high, fs=FS, order=3):
    x = finite_interp(x)
    ny = fs / 2.0
    high = min(high, ny * 0.95)
    sos = signal.butter(order, [low/ny, high/ny], btype="bandpass", output="sos")
    try:
        return signal.sosfiltfilt(sos, x)
    except Exception:
        return x

def basic_stats(x, prefix):
    x = np.asarray(x, dtype=float)
    return {
        f"{prefix}_mean": float(np.mean(x)),
        f"{prefix}_std": float(np.std(x)),
        f"{prefix}_median": float(np.median(x)),
        f"{prefix}_iqr": float(stats.iqr(x)),
        f"{prefix}_range": float(np.ptp(x)),
        f"{prefix}_rms": float(np.sqrt(np.mean(x*x))),
        f"{prefix}_mad": float(np.median(np.abs(x - np.median(x)))),
        f"{prefix}_skew": float(np.nan_to_num(stats.skew(x))),
        f"{prefix}_kurt": float(np.nan_to_num(stats.kurtosis(x))),
    }

def spectral_stats(x, prefix, fs=FS):
    f, p = signal.welch(x, fs=fs, nperseg=min(512, len(x)))
    den = np.sum(p) + 1e-12
    def band(lo, hi):
        return float(np.sum(p[(f>=lo) & (f<hi)]) / den)
    return {
        f"{prefix}_domfreq": float(f[np.argmax(p)]),
        f"{prefix}_spec_centroid": float(np.sum(f*p)/den),
        f"{prefix}_power_0p5_2": band(0.5,2.0),
        f"{prefix}_power_2_5": band(2.0,5.0),
        f"{prefix}_power_5_12": band(5.0,12.0),
    }

def detect_beats(x, modality, fs=FS):
    if modality == "ECG":
        distance = int(0.25 * fs)
        prom = max(0.35*np.std(x), 1e-8)
    else:
        distance = int(0.30 * fs)
        prom = max(0.25*np.std(x), 1e-8)
    peaks, _ = signal.find_peaks(x, distance=distance, prominence=prom)
    return peaks

def rhythm_features(x, prefix, modality, fs=FS):
    peaks = detect_beats(x, modality, fs)
    ibi = np.diff(peaks) / fs
    ibi = ibi[(ibi > 0.25) & (ibi < 2.5)]
    out = {
        f"{prefix}_peak_count": int(len(peaks)),
        f"{prefix}_ibi_n": int(len(ibi)),
        f"{prefix}_rate_bpm": float(60/np.mean(ibi)) if len(ibi) else np.nan,
        f"{prefix}_ibi_mean": float(np.mean(ibi)) if len(ibi) else np.nan,
        f"{prefix}_ibi_std": float(np.std(ibi)) if len(ibi) else np.nan,
        f"{prefix}_ibi_cv": float(np.std(ibi)/(np.mean(ibi)+1e-12)) if len(ibi) else np.nan,
        f"{prefix}_rmssd": float(np.sqrt(np.mean(np.diff(ibi)**2))) if len(ibi)>1 else np.nan,
        f"{prefix}_pnn50": float(np.mean(np.abs(np.diff(ibi)) > 0.05)) if len(ibi)>1 else np.nan,
    }
    if len(ibi) >= 4:
        hist = np.histogram(ibi, bins=8, density=False)[0] + 1e-12
        out[f"{prefix}_ibi_entropy"] = float(stats.entropy(hist))
    else:
        out[f"{prefix}_ibi_entropy"] = np.nan
    return out, peaks

def cross_features(ecg, ppg, ecg_peaks, ppg_peaks, fs=FS):
    e = (ecg - np.mean(ecg))/(np.std(ecg)+1e-8)
    p = (ppg - np.mean(ppg))/(np.std(ppg)+1e-8)
    cc = signal.correlate(p,e,mode="full",method="fft")
    lags = signal.correlation_lags(len(p),len(e),mode="full")
    m = (lags>=0) & (lags<=int(0.8*fs))
    lag = np.nan
    corr = np.nan
    if np.any(m):
        best = np.argmax(cc[m])
        lag = lags[m][best]/fs
        corr = cc[m][best]/(len(e)+1e-12)

    er = 60/(np.mean(np.diff(ecg_peaks))/fs) if len(ecg_peaks)>1 else np.nan
    pr = 60/(np.mean(np.diff(ppg_peaks))/fs) if len(ppg_peaks)>1 else np.nan

    return {
        "cross_ecg_ppg_lag_sec": float(lag) if np.isfinite(lag) else np.nan,
        "cross_maxcorr": float(corr) if np.isfinite(corr) else np.nan,
        "cross_rate_difference": float(er-pr) if np.isfinite(er) and np.isfinite(pr) else np.nan
    }

def feature_window(ecg_raw, ppg_raw):
    ecg = bandpass(ecg_raw,0.5,40)
    ppg = bandpass(ppg_raw,0.5,12)
    d={}
    d.update(basic_stats(ecg,"ecg"))
    d.update(spectral_stats(ecg,"ecg"))
    er, ep = rhythm_features(ecg,"ecg","ECG")
    d.update(er)

    d.update(basic_stats(ppg,"ppg"))
    d.update(spectral_stats(ppg,"ppg"))
    pr, pp = rhythm_features(ppg,"ppg","PPG")
    d.update(pr)

    d.update(cross_features(ecg,ppg,ep,pp))
    return d


In [ ]:

#@title 6. Extract balanced low-compute 30-second windows
rows=[]
win=WINDOW_SEC*FS
step=STEP_SEC*FS

for i,rec in usable_df.reset_index(drop=True).iterrows():
    p=Path(rec["path"])
    try:
        raw=pd.read_csv(p,usecols=[rec["ecg_col"],rec["ppg_col"]])
        ecg=pd.to_numeric(raw[rec["ecg_col"]],errors="coerce").to_numpy()
        ppg=pd.to_numeric(raw[rec["ppg_col"]],errors="coerce").to_numpy()

        n_possible=max(0,1+(min(len(ecg),len(ppg))-win)//step)
        if n_possible<=0:
            continue

        take=min(MAX_WINDOWS_PER_SUBJECT,n_possible)
        starts=np.linspace(0,n_possible-1,take,dtype=int)*step
        accepted=0

        for start in np.unique(starts):
            ew=ecg[start:start+win]
            pw=ppg[start:start+win]
            if len(ew)!=win or len(pw)!=win:
                continue
            if np.mean(np.isfinite(ew))<0.95 or np.mean(np.isfinite(pw))<0.95:
                continue

            feat=feature_window(ew,pw)
            if (not np.isfinite(feat.get("ecg_rate_bpm",np.nan))
                and not np.isfinite(feat.get("ppg_rate_bpm",np.nan))):
                continue

            feat.update({
                "subject":rec["subject"],
                "class_name":rec["class"],
                "label":int(rec["label"]),
                "start_sec":float(start/FS)
            })
            rows.append(feat)
            accepted+=1

        print(f"{i+1:02d}/{len(usable_df)} {rec['subject']}: {accepted} windows")

    except Exception as e:
        print("Skipped",p.name,":",e)

features=pd.DataFrame(rows)
assert len(features)>=300,(
    f"Only {len(features)} valid windows were extracted. "
    "Inspect dataset_audit.csv and the printed skipped-file messages."
)
assert features["subject"].nunique()>=25,"Too few subjects remained after feature extraction."

print("\nTotal windows:",len(features))
print("Subjects:",features["subject"].nunique())
display(features.groupby("class_name").agg(
    windows=("label","size"),
    subjects=("subject","nunique")
))
features.to_csv(OUT/"window_features.csv",index=False)


In [ ]:

#@title 7. Strict subject-wise train / validation / calibration / test split
y=features["label"].to_numpy()
groups=features["subject"].to_numpy()
idx=np.arange(len(features))

split_ok=False
for attempt in range(100):
    rs=SEED+attempt

    g_test=GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=rs)
    fit_idx,test_idx=next(g_test.split(idx,y,groups))

    g_cal=GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=rs+1000)
    trainval_rel,cal_rel=next(g_cal.split(fit_idx,y[fit_idx],groups[fit_idx]))
    trainval_idx=fit_idx[trainval_rel]
    cal_idx=fit_idx[cal_rel]

    g_val=GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=rs+2000)
    train_rel,val_rel=next(g_val.split(trainval_idx,y[trainval_idx],groups[trainval_idx]))
    train_idx=trainval_idx[train_rel]
    val_idx=trainval_idx[val_rel]

    splits=[train_idx,val_idx,cal_idx,test_idx]
    if all(set(np.unique(y[s]))=={0,1} for s in splits):
        split_ok=True
        break

assert split_ok,"Unable to create a four-way subject-wise split containing both classes."

split_subjects=[set(groups[s]) for s in splits]
for a in range(4):
    for b in range(a+1,4):
        assert split_subjects[a].isdisjoint(split_subjects[b]),"Subject leakage detected."

summary=[]
for name,s in zip(["train","validation","calibration","test"],splits):
    temp=features.iloc[s]
    summary.append({
        "split":name,
        "windows":len(s),
        "subjects":temp["subject"].nunique(),
        "AF_subjects":temp[temp.label==1]["subject"].nunique(),
        "NonAF_subjects":temp[temp.label==0]["subject"].nunique(),
        "AF_windows":int((temp.label==1).sum()),
        "NonAF_windows":int((temp.label==0).sum())
    })

split_df=pd.DataFrame(summary)
display(split_df)
split_df.to_csv(OUT/"subjectwise_split.csv",index=False)


In [ ]:

#@title 8. Feature sets, models and evaluation utilities
meta_cols=["subject","class_name","label","start_sec"]
feature_cols=[c for c in features.columns if c not in meta_cols]
ecg_cols=[c for c in feature_cols if c.startswith("ecg_")]
ppg_cols=[c for c in feature_cols if c.startswith("ppg_")]
fusion_cols=feature_cols

def make_models():
    return {
        "ExtraTrees":ExtraTreesClassifier(
            n_estimators=350,max_features="sqrt",min_samples_leaf=2,
            class_weight="balanced",random_state=SEED,n_jobs=-1
        ),
        "RandomForest":RandomForestClassifier(
            n_estimators=300,max_depth=12,min_samples_leaf=2,max_features="sqrt",
            class_weight="balanced",random_state=SEED,n_jobs=-1
        ),
        "HistGradientBoosting":HistGradientBoostingClassifier(
            max_iter=180,learning_rate=0.06,max_leaf_nodes=31,
            l2_regularization=1.0,random_state=SEED
        )
    }

def ece(y_true,prob,bins=10):
    conf=np.maximum(prob,1-prob)
    pred=(prob>=0.5).astype(int)
    correct=(pred==y_true).astype(float)
    out=0.0
    for lo,hi in zip(np.linspace(.5,1,bins+1)[:-1],np.linspace(.5,1,bins+1)[1:]):
        m=(conf>lo)&(conf<=hi)
        if np.any(m):
            out+=np.mean(m)*abs(np.mean(correct[m])-np.mean(conf[m]))
    return float(out)

def specificity(y_true,pred):
    tn,fp,fn,tp=confusion_matrix(y_true,pred,labels=[0,1]).ravel()
    return float(tn/(tn+fp)) if tn+fp else np.nan

def metrics(name,y_true,prob):
    pred=(prob>=0.5).astype(int)
    return {
        "Model":name,
        "Accuracy":accuracy_score(y_true,pred),
        "BalancedAccuracy":balanced_accuracy_score(y_true,pred),
        "Precision":precision_score(y_true,pred,zero_division=0),
        "Sensitivity":recall_score(y_true,pred,zero_division=0),
        "Specificity":specificity(y_true,pred),
        "F1":f1_score(y_true,pred,zero_division=0),
        "AUROC":roc_auc_score(y_true,prob),
        "AUPRC":average_precision_score(y_true,prob),
        "Brier":brier_score_loss(y_true,prob),
        "ECE":ece(y_true,prob),
        "LogLoss":log_loss(y_true,np.c_[1-prob,prob],labels=[0,1])
    }

def fit_modality(cols,modality):
    imputer=SimpleImputer(strategy="median",add_indicator=True)
    Xtr=imputer.fit_transform(features.iloc[train_idx][cols])
    Xva=imputer.transform(features.iloc[val_idx][cols])
    Xca=imputer.transform(features.iloc[cal_idx][cols])
    Xte=imputer.transform(features.iloc[test_idx][cols])

    fitted={}
    pva,pca,pte={},{},{}
    for name,model in make_models().items():
        model.fit(Xtr,y[train_idx])
        fitted[name]=model
        pva[name]=model.predict_proba(Xva)[:,1]
        pca[name]=model.predict_proba(Xca)[:,1]
        pte[name]=model.predict_proba(Xte)[:,1]

    scores=np.array([
        max(f1_score(y[val_idx],(pva[n]>=.5).astype(int),zero_division=0),1e-4)
        for n in fitted
    ])
    weights=scores/scores.sum()

    return {
        "modality":modality,
        "cols":cols,
        "imputer":imputer,
        "models":fitted,
        "weights":weights,
        "fused_val":sum(w*pva[n] for w,n in zip(weights,fitted)),
        "fused_cal":sum(w*pca[n] for w,n in zip(weights,fitted)),
        "fused_test":sum(w*pte[n] for w,n in zip(weights,fitted)),
        "p_test":pte
    }


In [ ]:

#@title 9. ECG-only, PPG-only and ECG+PPG multimodal experiments
objects={}
result_rows=[]

for modality,cols in {
    "ECG_only":ecg_cols,
    "PPG_only":ppg_cols,
    "ECG_PPG_Fusion":fusion_cols
}.items():
    print("Training",modality,"|",len(cols),"features")
    obj=fit_modality(cols,modality)
    objects[modality]=obj

    for name,p in obj["p_test"].items():
        result_rows.append(metrics(f"{modality}_{name}",y[test_idx],p))
    result_rows.append(metrics(f"{modality}_WeightedFusion",y[test_idx],obj["fused_test"]))

model_results=pd.DataFrame(result_rows).sort_values("AUROC",ascending=False)
display(model_results.round(4))
model_results.to_csv(OUT/"model_modality_results.csv",index=False)

fusion_obj=objects["ECG_PPG_Fusion"]
print("Fusion learner weights:")
for n,w in zip(fusion_obj["models"],fusion_obj["weights"]):
    print(f" {n}: {w:.3f}")


In [ ]:

#@title 10. Temperature calibration on a separate calibration split
p_cal_raw=np.clip(fusion_obj["fused_cal"],1e-6,1-1e-6)
p_test_raw=np.clip(fusion_obj["fused_test"],1e-6,1-1e-6)

def temp_binary(prob,T):
    logit=np.log(prob/(1-prob))
    return expit(logit/max(float(T),1e-6))

opt=minimize_scalar(
    lambda T:log_loss(
        y[cal_idx],
        np.c_[1-temp_binary(p_cal_raw,T),temp_binary(p_cal_raw,T)],
        labels=[0,1]
    ),
    bounds=(0.25,5.0),
    method="bounded"
)

T=float(opt.x)
p_cal=temp_binary(p_cal_raw,T)
p_test=temp_binary(p_test_raw,T)

cal_df=pd.DataFrame([
    metrics("RawFusion",y[test_idx],p_test_raw),
    metrics("TemperatureCalibratedFusion",y[test_idx],p_test)
])
display(cal_df.round(4))
print("Optimal temperature:",round(T,4))
cal_df.to_csv(OUT/"calibration_results.csv",index=False)


In [ ]:

#@title 11. Split-conformal prediction sets
cal_true_prob=np.where(y[cal_idx]==1,p_cal,1-p_cal)
scores=1-cal_true_prob

def higher_quantile(x,q):
    try:
        return float(np.quantile(x,q,method="higher"))
    except TypeError:
        return float(np.quantile(x,q,interpolation="higher"))

conf_rows=[]
set_store={}

for alpha in [0.05,0.10]:
    n=len(scores)
    qlevel=min(1.0,math.ceil((n+1)*(1-alpha))/n)
    qhat=higher_quantile(scores,qlevel)

    allow0=(1-p_test)>=(1-qhat)
    allow1=p_test>=(1-qhat)
    sets=np.c_[allow0,allow1]

    empty=np.where(sets.sum(axis=1)==0)[0]
    if len(empty):
        hard=(p_test[empty]>=.5).astype(int)
        sets[empty,:]=False
        sets[empty,hard]=True

    coverage=np.mean(sets[np.arange(len(test_idx)),y[test_idx]])
    size=sets.sum(axis=1)

    conf_rows.append({
        "NominalCoverage":1-alpha,
        "EmpiricalCoverage":coverage,
        "MeanPredictionSetSize":float(np.mean(size)),
        "SingletonRate":float(np.mean(size==1)),
        "ConformalQuantile":qhat
    })
    set_store[alpha]=sets

conformal_df=pd.DataFrame(conf_rows)
display(conformal_df.round(4))
conformal_df.to_csv(OUT/"conformal_results.csv",index=False)


In [ ]:

#@title 12. Reliability-guided edge-to-cloud escalation
confidence=np.maximum(p_test,1-p_test)
pred=(p_test>=.5).astype(int)
policy_rows=[]

for threshold in [0.60,0.70,0.80,0.90,0.95]:
    local=confidence>=threshold
    policy_rows.append({
        "ConfidenceThreshold":threshold,
        "LocalDecisionRate":float(np.mean(local)),
        "EscalationRate":float(1-np.mean(local)),
        "LocalAccuracy":accuracy_score(y[test_idx][local],pred[local]) if np.any(local) else np.nan,
        "LocalF1":f1_score(y[test_idx][local],pred[local],zero_division=0) if np.any(local) else np.nan
    })

policy_df=pd.DataFrame(policy_rows)
display(policy_df.round(4))
policy_df.to_csv(OUT/"edge_cloud_policy.csv",index=False)


In [ ]:

#@title 13. Noise and missing-sensor robustness
def fusion_predict_from_frame(Xframe):
    Z=fusion_obj["imputer"].transform(Xframe[fusion_obj["cols"]])
    out=np.zeros(len(Xframe),dtype=float)
    for w,(name,model) in zip(fusion_obj["weights"],fusion_obj["models"].items()):
        out+=w*model.predict_proba(Z)[:,1]
    return temp_binary(np.clip(out,1e-6,1-1e-6),T)

Xtest=features.iloc[test_idx][fusion_cols].copy().reset_index(drop=True)
train_sd=features.iloc[train_idx][fusion_cols].std().fillna(0).to_numpy()
rng=np.random.default_rng(SEED)

rob_rows=[metrics("Clean",y[test_idx],fusion_predict_from_frame(Xtest))]

for sigma in [0.05,0.10,0.20]:
    Xn=Xtest.copy()
    Xn[:]=Xn.to_numpy(dtype=float)+rng.normal(size=Xn.shape)*train_sd*sigma
    rob_rows.append(metrics(f"Noise_{sigma}",y[test_idx],fusion_predict_from_frame(Xn)))

Xm=Xtest.copy()
Xm[ecg_cols]=np.nan
rob_rows.append(metrics("ECG_missing",y[test_idx],fusion_predict_from_frame(Xm)))

Xm=Xtest.copy()
Xm[ppg_cols]=np.nan
rob_rows.append(metrics("PPG_missing",y[test_idx],fusion_predict_from_frame(Xm)))

robust_df=pd.DataFrame(rob_rows)
display(robust_df.round(4))
robust_df.to_csv(OUT/"robustness_results.csv",index=False)


In [ ]:

#@title 14. Permutation-based feature attribution
best_name=list(fusion_obj["models"].keys())[int(np.argmax(fusion_obj["weights"]))]

pipe=Pipeline([
    ("imputer",SimpleImputer(strategy="median",add_indicator=True)),
    ("model",make_models()[best_name])
])
pipe.fit(features.iloc[train_idx][fusion_cols],y[train_idx])

perm=permutation_importance(
    pipe,
    features.iloc[test_idx][fusion_cols],
    y[test_idx],
    scoring="roc_auc",
    n_repeats=5,
    random_state=SEED,
    n_jobs=-1
)

importance_df=pd.DataFrame({
    "Feature":fusion_cols,
    "ImportanceMean":perm.importances_mean,
    "ImportanceStd":perm.importances_std
}).sort_values("ImportanceMean",ascending=False)

display(importance_df.head(20).round(5))
importance_df.to_csv(OUT/"feature_importance.csv",index=False)

top=importance_df.head(20).sort_values("ImportanceMean")
fig,ax=plt.subplots(figsize=(8,7))
ax.barh(top["Feature"],top["ImportanceMean"])
ax.set_xlabel("Permutation decrease in AUROC")
ax.set_title("Top multimodal predictors")
fig.tight_layout()
fig.savefig(OUT/"feature_importance.png",dpi=220,bbox_inches="tight")
plt.show()


In [ ]:

#@title 15. Subject-level bootstrap 95% confidence intervals
test_subjects=features.iloc[test_idx]["subject"].to_numpy()
unique_test_subjects=np.unique(test_subjects)
yt=y[test_idx]

def subject_bootstrap_metric(metric_fn,B=BOOTSTRAPS):
    rng=np.random.default_rng(SEED)
    vals=[]
    for _ in range(B):
        sampled=rng.choice(unique_test_subjects,size=len(unique_test_subjects),replace=True)
        positions=[]
        for s in sampled:
            positions.extend(np.where(test_subjects==s)[0].tolist())
        positions=np.array(positions,dtype=int)
        yy=yt[positions]
        pp=p_test[positions]
        if len(np.unique(yy))<2:
            continue
        vals.append(metric_fn(yy,pp))
    vals=np.asarray(vals)
    return float(np.mean(vals)),float(np.quantile(vals,.025)),float(np.quantile(vals,.975))

ci_rows=[]
metric_fns={
    "AUROC":lambda yy,pp:roc_auc_score(yy,pp),
    "AUPRC":lambda yy,pp:average_precision_score(yy,pp),
    "F1":lambda yy,pp:f1_score(yy,(pp>=.5).astype(int),zero_division=0),
    "BalancedAccuracy":lambda yy,pp:balanced_accuracy_score(yy,(pp>=.5).astype(int))
}

for name,fn in metric_fns.items():
    mean,lo,hi=subject_bootstrap_metric(fn)
    ci_rows.append({"Metric":name,"BootstrapMean":mean,"CI95_Low":lo,"CI95_High":hi})

ci_df=pd.DataFrame(ci_rows)
display(ci_df.round(4))
ci_df.to_csv(OUT/"subject_bootstrap_CI.csv",index=False)


In [ ]:

#@title 16. McNemar significance test: multimodal fusion vs strongest unimodal model
unimodal_candidates={
    "ECG_only":objects["ECG_only"]["fused_test"],
    "PPG_only":objects["PPG_only"]["fused_test"]
}

best_unimodal=max(
    unimodal_candidates,
    key=lambda k:roc_auc_score(yt,unimodal_candidates[k])
)

base_pred=(unimodal_candidates[best_unimodal]>=.5).astype(int)
fusion_pred=(p_test>=.5).astype(int)

base_correct=base_pred==yt
fusion_correct=fusion_pred==yt

b=int(np.sum((~base_correct)&fusion_correct))
c=int(np.sum(base_correct&(~fusion_correct)))

p_mc=1.0 if b+c==0 else float(
    stats.binomtest(min(b,c),n=b+c,p=.5,alternative="two-sided").pvalue
)

sig_df=pd.DataFrame([{
    "Comparison":f"Calibrated multimodal fusion vs {best_unimodal}",
    "FusionCorrect_BaseWrong":b,
    "BaseCorrect_FusionWrong":c,
    "McNemarExactP":p_mc
}])
display(sig_df)
sig_df.to_csv(OUT/"mcnemar_significance.csv",index=False)


In [ ]:

#@title 17. Confusion matrix, classification report and ROC/PR figures
pred=(p_test>=.5).astype(int)

report_df=pd.DataFrame(
    classification_report(
        yt,pred,labels=[0,1],target_names=["Non-AF","AF"],
        output_dict=True,zero_division=0
    )
).T
display(report_df.round(4))
report_df.to_csv(OUT/"classification_report.csv")

cm=confusion_matrix(yt,pred,labels=[0,1])

fig,ax=plt.subplots(figsize=(5,4))
im=ax.imshow(cm)
fig.colorbar(im,ax=ax)
for i in range(2):
    for j in range(2):
        ax.text(j,i,str(cm[i,j]),ha="center",va="center")
ax.set_xticks([0,1]); ax.set_xticklabels(["Non-AF","AF"])
ax.set_yticks([0,1]); ax.set_yticklabels(["Non-AF","AF"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Subject-wise test confusion matrix")
fig.tight_layout()
fig.savefig(OUT/"confusion_matrix.png",dpi=220,bbox_inches="tight")
plt.show()

from sklearn.metrics import RocCurveDisplay,PrecisionRecallDisplay

fig,ax=plt.subplots(figsize=(5,4))
RocCurveDisplay.from_predictions(yt,p_test,ax=ax,name="CardioNexus")
ax.set_title("ROC curve")
fig.tight_layout()
fig.savefig(OUT/"roc_curve.png",dpi=220,bbox_inches="tight")
plt.show()

fig,ax=plt.subplots(figsize=(5,4))
PrecisionRecallDisplay.from_predictions(yt,p_test,ax=ax,name="CardioNexus")
ax.set_title("Precision–Recall curve")
fig.tight_layout()
fig.savefig(OUT/"pr_curve.png",dpi=220,bbox_inches="tight")
plt.show()


In [ ]:

#@title 18. Edge inference latency, throughput and serialized model size
profile_X=features.iloc[test_idx][fusion_cols].head(min(300,len(test_idx))).copy()
Z=fusion_obj["imputer"].transform(profile_X)

profile_rows=[]
for name,model in fusion_obj["models"].items():
    _=model.predict_proba(Z[:min(10,len(Z))])
    repeats=30
    t0=time.perf_counter()
    for _ in range(repeats):
        _=model.predict_proba(Z)
    elapsed=(time.perf_counter()-t0)/repeats

    path=OUT/f"{name}_model.joblib"
    joblib.dump(model,path,compress=3)

    profile_rows.append({
        "Model":name,
        "ProfileWindows":len(Z),
        "BatchLatency_ms":elapsed*1000,
        "Latency_ms_per_window":elapsed*1000/max(len(Z),1),
        "Throughput_windows_per_sec":len(Z)/elapsed,
        "ModelSize_MB":path.stat().st_size/(1024**2)
    })

joblib.dump(fusion_obj["imputer"],OUT/"fusion_imputer.joblib",compress=3)
joblib.dump(
    {"weights":fusion_obj["weights"],"feature_cols":fusion_cols,"temperature":T},
    OUT/"fusion_metadata.joblib",
    compress=3
)

profile_df=pd.DataFrame(profile_rows)
display(profile_df.round(4))
profile_df.to_csv(OUT/"edge_profile.csv",index=False)


In [ ]:

#@title 19. Final ablation table and test predictions
ablation=pd.DataFrame([
    metrics("ECG only",yt,objects["ECG_only"]["fused_test"]),
    metrics("PPG only",yt,objects["PPG_only"]["fused_test"]),
    metrics("ECG+PPG raw fusion",yt,p_test_raw),
    metrics("CardioNexus calibrated fusion",yt,p_test)
])
display(ablation.round(4))
ablation.to_csv(OUT/"master_ablation.csv",index=False)

sets90=set_store[0.10]
pred_df=features.iloc[test_idx][["subject","start_sec","class_name","label"]].copy().reset_index(drop=True)
pred_df["AF_probability"]=p_test
pred_df["prediction"]=pred
pred_df["prediction_name"]=np.where(pred==1,"AF","Non-AF")
pred_df["confidence"]=np.maximum(p_test,1-p_test)
pred_df["conformal_set_size_90"]=sets90.sum(axis=1)
pred_df["conformal_set_90"]=[
    ("Non-AF" if s[0] else "")+(" | " if s[0] and s[1] else "")+("AF" if s[1] else "")
    for s in sets90
]
pred_df.to_csv(OUT/"test_predictions.csv",index=False)
display(pred_df.head())


In [ ]:

#@title 20. Reproducibility manifest, required-output check and downloadable ZIP
manifest={
    "title":"CardioNexus: Multimodal ECG–PPG Intelligence for Reliable Real-Time Atrial Fibrillation Screening at the Edge",
    "dataset":"MIMIC PERform AF",
    "source":"Zenodo record 6807403",
    "endpoint":"AF vs non-AF",
    "sampling_rate_hz":FS,
    "window_sec":WINDOW_SEC,
    "step_sec":STEP_SEC,
    "max_windows_per_subject":MAX_WINDOWS_PER_SUBJECT,
    "seed":SEED,
    "bootstrap_replicates":BOOTSTRAPS,
    "subjects_total":int(features["subject"].nunique()),
    "windows_total":int(len(features)),
    "temperature":float(T),
    "models":list(fusion_obj["models"].keys()),
    "fusion_weights":{
        n:float(w) for n,w in zip(fusion_obj["models"].keys(),fusion_obj["weights"])
    },
    "scientific_limit":"AF screening only; not a general cardiovascular disease diagnosis model"
}

with open(OUT/"manifest.json","w") as f:
    json.dump(manifest,f,indent=2)

required=[
    "dataset_audit.csv","window_features.csv","subjectwise_split.csv",
    "model_modality_results.csv","calibration_results.csv","conformal_results.csv",
    "edge_cloud_policy.csv","robustness_results.csv","feature_importance.csv",
    "subject_bootstrap_CI.csv","mcnemar_significance.csv","classification_report.csv",
    "edge_profile.csv","master_ablation.csv","test_predictions.csv","manifest.json"
]

missing=[x for x in required if not (OUT/x).exists()]
assert not missing,f"Mandatory outputs missing: {missing}"

zip_path=Path("/content/CardioNexus_MIMIC_PERform_AF_RESULTS.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p,arcname=p.relative_to(OUT))

print("FULL PIPELINE COMPLETE")
print("Result ZIP:",zip_path)
print("ZIP size (MB):",round(zip_path.stat().st_size/1e6,2))
for p in sorted(OUT.iterdir()):
    print(" -",p.name)

from google.colab import files
files.download(str(zip_path))



## Reporting guidance

Use the wording **AF screening / AF detection**, not broad cardiovascular disease diagnosis.

Primary manuscript comparison:

**ECG-only → PPG-only → ECG+PPG multi-learner fusion → calibrated CardioNexus**

Report only the actual outputs generated by the notebook. Do not invent performance values.
